# Example 12.6.2 / 1: Solve a shared steel plan

Companion notebook for **12.6.2 Single-Stage Stochastic Models** in the AMPL lab Documentation, section "Solve a shared steel plan". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve a shared steel plan

The steel model of Model Components (AMPL lab documentation, section 1) becomes single-stage when profit per ton and available
hours depend on three scenarios. `ampl.solve`, `to_dict()`, `value()` and `get_values("slack")`
read the result.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                               # steel products
set SCENARIOS;                          # market outcomes
param probability {SCENARIOS} >= 0;     # scenario weight
param rate {PROD} > 0;                  # tons per hour
param market {PROD} >= 0;               # tons
param profit {PROD, SCENARIOS};         # dollars per ton
param avail {SCENARIOS} >= 0;           # hours

# One shared production plan for every scenario.
var Make {p in PROD} >= 0 <= market[p];

# Profit the shared plan earns in each scenario.
var ScenarioProfit {s in SCENARIOS} =
    sum {p in PROD} profit[p,s] * Make[p];

maximize ExpectedProfit:                # probability-weighted profit, dollars
    sum {s in SCENARIOS} probability[s] * ScenarioProfit[s];

# The plan must fit the hours of every scenario.
subject to TimeEveryScenario {s in SCENARIOS}:
    sum {p in PROD} Make[p] / rate[p] <= avail[s];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.set["SCENARIOS"] = ["weak", "base", "strong"]
ampl.param["probability"] = {"weak": 0.3, "base": 0.4, "strong": 0.3}
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["market"] = {"bands": 6000, "coils": 4000}
ampl.param["profit"] = {
    ("bands", "weak"): 20, ("bands", "base"): 25, ("bands", "strong"): 28,
    ("coils", "weak"): 24, ("coils", "base"): 30, ("coils", "strong"): 36,
}
ampl.param["avail"] = {"weak": 30, "base": 40, "strong": 45}

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
print(ampl.var["Make"].to_dict())
print(ampl.var["ScenarioProfit"].to_dict())
print(ampl.obj["ExpectedProfit"].value())
print(ampl.con["TimeEveryScenario"].get_values("slack").to_dict())

solved
{'bands': 6000, 'coils': 0}
{'base': 150000, 'strong': 168000, 'weak': 120000}
146400.0
{'base': 10, 'strong': 15, 'weak': 0}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`; `solve` and `display` replace the amplpy calls:

In [6]:
%%writefile data.dat
set PROD := bands coils;
set SCENARIOS := weak base strong;
param probability := weak 0.3  base 0.4  strong 0.3;
param:   rate  market :=
  bands   200    6000
  coils   140    4000;
param profit:  weak  base  strong :=
  bands          20    25      28
  coils          24    30      36;
param avail := weak 30  base 40  strong 45;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Make;
display ScenarioProfit;
display ExpectedProfit;
display TimeEveryScenario.slack;

HiGHS 1.14.0: optimal solution; objective 146400
1 simplex iterations
0 barrier iterations
Make [*] :=
bands  6000
coils     0
;

ScenarioProfit [*] :=
  base  150000
strong  168000
  weak  120000
;

ExpectedProfit = 146400

TimeEveryScenario.slack [*] :=
  base  10
strong  15
  weak   0
;

